In [1]:
import requests

es = "http://elasticsearch:9200"
r = requests.get(f"{es}/_cat/indices/api-events-*?v")
print(r.text)

health status index                 uuid                   pri rep docs.count docs.deleted store.size pri.store.size dataset.size
yellow open   api-events-2026.02.24 PO7hYk1ZQDSZSInd7odLwQ   1   1         38            0    125.3kb        125.3kb      125.3kb



In [2]:
import requests, pandas as pd

es = "http://elasticsearch:9200"
index = "api-events-*"

query = {
  "size": 1000,
  "query": {"match_all": {}}
}

resp = requests.get(f"{es}/{index}/_search", json=query).json()
hits = [h["_source"] for h in resp["hits"]["hits"]]
df = pd.DataFrame(hits)
df.head()

,timestamp,event,temp,source,city,@timestamp,description,@version
0,2026-02-24T23:00,"{'original': '{""city"": ""Lyon"", ""temp"": 10.1, ""...",10.1,open-meteo,Lyon,2026-02-24T23:00:00.000Z,live_weather,1
1,2026-02-24T23:00,"{'original': '{""city"": ""Paris"", ""temp"": 10.7, ...",10.7,open-meteo,Paris,2026-02-24T23:00:00.000Z,live_weather,1
2,2026-02-24T23:00,"{'original': '{""city"": ""Lyon"", ""temp"": 10.1, ""...",10.1,open-meteo,Lyon,2026-02-24T23:00:00.000Z,live_weather,1
3,2026-02-24T23:00,"{'original': '{""city"": ""Paris"", ""temp"": 10.7, ...",10.7,open-meteo,Paris,2026-02-24T23:00:00.000Z,live_weather,1
4,2026-02-24T23:00,"{'original': '{""city"": ""Lyon"", ""temp"": 10.1, ""...",10.1,open-meteo,Lyon,2026-02-24T23:00:00.000Z,live_weather,1


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

spark = SparkSession.builder.appName("api-events").getOrCreate()

sdf = spark.createDataFrame(df)
sdf.groupBy("city").agg(avg("temp").alias("temp_moy")).show()

+-----+------------------+
| city|          temp_moy|
+-----+------------------+
|Paris|              10.7|
| Lyon|10.105882352941176|
+-----+------------------+



In [5]:
from pyspark.sql.functions import to_timestamp, window

sdf2 = sdf.withColumn("ts", to_timestamp("timestamp"))
sdf2.groupBy(window(col("ts"), "1 hour"), col("city")) \
    .agg(avg("temp").alias("temp_moy")) \
    .orderBy("window") \
    .show(truncate=False)

+------------------------------------------+-----+------------------+
|window                                    |city |temp_moy          |
+------------------------------------------+-----+------------------+
|{2026-02-24 22:00:00, 2026-02-24 23:00:00}|Paris|10.7              |
|{2026-02-24 22:00:00, 2026-02-24 23:00:00}|Lyon |10.2              |
|{2026-02-24 23:00:00, 2026-02-25 00:00:00}|Lyon |10.1              |
|{2026-02-24 23:00:00, 2026-02-25 00:00:00}|Paris|10.700000000000001|
+------------------------------------------+-----+------------------+

